# Attention Mechanisms aur Transformers

Recurrent Networks ki ek badi kamzori yeh hai ki sequence ke saare words ka result par same impact hota hai. Isse Named Entity Recognition aur Machine Translation jaise sequence-to-sequence tasks ke liye standard LSTM encoder-decoder models ke saath sub-optimal performance milti hai. Asal mein, input sequence ke kuch specific words ka sequential output par doosre words ke muqabale zyada impact hota hai.

Machine translation jaise sequence-to-sequence model par vichaar karein. Ise do recurrent networks se implement kiya jaata hai, jahan ek network (**encoder**) input sequence ko ek hidden state mein summarize karta hai, aur doosra network (**decoder**) is hidden state ko translated result mein badalta hai. Is approach ki problem yeh hai ki network ki final state ko sentence ki shuruaat yaad rakhne mein dikkat hoti hai, jisse lambe sentences par model ki quality kharab ho jaati hai.

**Attention Mechanisms** har input vector ke contextual impact ko RNN ke har output prediction par weigh karne ka ek tarika dete hain. Ise implement karne ka tarika yeh hai ki input RNN ki intermediate states aur output RNN ke beech shortcuts banaye jaate hain. Is tarah, jab output symbol $y_t$ generate kiya jaata hai, toh hum saari input hidden states $h_i$ ko alag-alag weight coefficients $\alpha_{t,i}$ ke saath dhyan mein rakhenge.

![Additive attention layer wala encoder/decoder model](../../../../../lessons/5-NLP/18-Transformers/images/encoder-decoder-attention.png)
*[Bahdanau et al., 2015](https://arxiv.org/pdf/1409.0473.pdf) mein additive attention mechanism ke saath encoder-decoder model, [is blog post](https://lilianweng.github.io/lil-log/2018/06/24/attention-attention.html) se liya gaya*

Attention matrix $\{\alpha_{i,j}\}$ yeh dikhayega ki output sequence mein kisi diye gaye word ko generate karne mein kaun se input words kitni role nibhate hain. Neeche is tarah ki matrix ka ek example diya gaya hai:

![RNNsearch-50 dwara paye gaye ek sample alignment ko dikhane wali image, Bahdanau - arviz.org se li gayi](../../../../../lessons/5-NLP/18-Transformers/images/bahdanau-fig3.png)

*[Bahdanau et al., 2015](https://arxiv.org/pdf/1409.0473.pdf) (Fig.3) se li gayi figure*

Attention mechanisms Natural Language Processing mein current ya near-current state-of-the-art ke liye zimmedaar hain. Lekin, attention add karne se model ke parameters ki sankhya kaafi badh jaati hai, jisse RNNs ke saath scaling problems paida hoti hain. RNNs ko scale karne mein ek major bottleneck yeh hai ki model ki recurrent nature training ko batch aur parallelize karna challenging bana deti hai. RNN mein sequence ke har element ko sequential order mein process karna padta hai, jiska matlab hai ki ise aasaani se parallelize nahi kiya ja sakta.

Attention mechanisms ko apnaye jaane aur is bottleneck ne un state-of-the-art Transformer models ke banne ka raasta khola, jinhein hum aaj BERT se lekar OpenGPT3 tak use karte hain.

## Transformer Models

Pichhli prediction ke context ko agle evaluation step mein forward karne ke bajaye, **Transformer models** **positional encodings** aur **attention** ka use karte hain taaki diye gaye input ke context ko ek specified text window ke andar capture kiya ja sake. Neeche di gayi image dikhati hai ki positional encodings aur attention ka use karke kisi window ke andar context kaise capture kiya ja sakta hai.

![Transformer models mein evaluation kaise kiya jaata hai, yeh dikhane wala animated GIF.](../../../../../lessons/5-NLP/18-Transformers/images/transformer-animated-explanation.gif)

Kyunki har input position ko independently har output position par map kiya jaata hai, Transformers RNNs ke muqabale behtar parallelism de sakte hain, jisse bade aur zyada expressive language models possible hote hain. Har attention head ka use words ke beech alag-alag relationships seekhne ke liye kiya ja sakta hai, jo downstream Natural Language Processing tasks ko improve karta hai.

## Ek Simple Transformer Model banana

Keras mein built-in Transformer layer nahi hai, lekin hum apni khud ki bana sakte hain. Pehle ki tarah, hum AG News dataset ki text classification par focus karenge, lekin yeh batana important hai ki Transformer models zyada mushkil NLP tasks mein best results dikhate hain.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

ds_train, ds_test = tfds.load('ag_news_subset').values()

def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

Keras mein nayi layers ko `Layer` class ko subclass karna chahiye aur `call` method implement karna chahiye. Chalo **Positional Embedding** layer se shuru karte hain. Hum [official Keras documentation](https://keras.io/examples/nlp/text_classification_with_transformer/) se kuch code use karenge. Hum maan lenge ki hum saari input sequences ko `maxlen` length tak pad karte hain.

In [ ]:
class TokenAndPositionEmbedding(keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
        self.maxlen = maxlen

    def call(self, x):
        maxlen = self.maxlen
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x+positions

Yeh layer do `Embedding` layers se bani hoti hai: ek token ko embed karne ke liye (jaisa humne pehle discuss kiya hai) aur doosri token ki position ko embed karne ke liye. Token ki position ko 0 se `maxlen` tak natural numbers ki sequence ke roop mein `tf.range` use karke banaya jaata hai, aur phir ise embedding layer mein pass kiya jaata hai. Uske baad, dono milne wale embedding vectors ko add kiya jaata hai, jisse input ka positionally-embedded representation taiyaar hota hai, jiska shape `maxlen`$\times$`embed_dim` hota hai.

Ab, chalo Transformer block implement karte hain. Yeh pehle define ki gayi embedding layer ka output lega:

In [ ]:
class TransformerBlock(keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, name='attn')
        self.ffn = keras.Sequential(
            [keras.layers.Dense(ff_dim, activation="relu"), keras.layers.Dense(embed_dim),]
        )
        self.layernorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = keras.layers.Dropout(rate)
        self.dropout2 = keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

Ab, hum poora Transformer model define karne ke liye taiyaar hain:

In [ ]:
embed_dim = 32  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network inside transformer
maxlen = 256
vocab_size = 20000

model = keras.models.Sequential([
    keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size,output_sequence_length=maxlen, input_shape=(1,)),
    TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim),
    TransformerBlock(embed_dim, num_heads, ff_dim),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dropout(0.1),
    keras.layers.Dense(20, activation="relu"),
    keras.layers.Dropout(0.1),
    keras.layers.Dense(4, activation="softmax")
])

model.summary()

In [ ]:
print('Training tokenizer')
model.layers[0].adapt(ds_train.map(extract_text))
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(128),validation_data=ds_test.map(tupelize).batch(128))

## BERT Transformer Models

**BERT** (Bidirectional Encoder Representations from Transformers) ek bahut bada multi-layer Transformer network hai, jisme *BERT-base* ke liye 12 layers aur *BERT-large* ke liye 24 layers hoti hain. Is model ko pehle bade text data (WikiPedia + books) ke corpus par unsupervised training (ek sentence mein hide kiye gaye words ko predict karna) use karke pre-train kiya jaata hai. Pre-training ke dauran, model language ko samajhne ki ek important ability develop karta hai, jise phir doosre datasets ke saath fine-tuning ke zariye use kiya ja sakta hai. Is process ko **Transfer Learning** kehte hain.

![http://jalammar.github.io/illustrated-bert/ se li gayi image](../../../../../lessons/5-NLP/18-Transformers/images/jalammarBERT-language-modeling-masked-lm.png)

BERT, DistilBERT, BigBird, OpenGPT3 aur doosre jaise Transformer architectures ke kai variants hain, jinhein fine-tune kiya ja sakta hai.

Chalo dekhte hain ki hum pre-trained BERT model use karke apni traditional sequence classification problem kaise solve kar sakte hain. Hum [official documentation](https://www.tensorflow.org/text/tutorials/classify_text_with_bert) se idea aur kuch code udhaar lenge.

Pre-trained models load karne ke liye, hum **Tensorflow hub** use karenge. Sabse pehle, chalo BERT-specific vectorizer load karte hain:

In [ ]:
import tensorflow_text 
import tensorflow_hub as hub
vectorizer = hub.KerasLayer('https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3')

In [ ]:
vectorizer(['I love transformers'])

Yeh zaroori hai ki aap wahi vectorizer use karein jo original network par training ke dauran use kiya gaya tha. Saath hi, BERT vectorizer teen components return karta hai:
* `input_word_ids`, jo input sentence ke liye token numbers ki sequence hai
* `input_mask`, jo dikhata hai ki sequence ka kaun sa hissa asli input hai aur kaun sa padding hai. Yeh `Masking` layer dwara banaye gaye mask jaisa hi hai
* `input_type_ids` language modeling tasks ke liye use hota hai, aur ek sequence mein do input sentences specify karne deta hai.

Iske baad, hum BERT feature extractor ko instantiate kar sakte hain:

In [ ]:
bert = hub.KerasLayer('https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-128_A-2/1')

In [ ]:
z = bert(vectorizer(['I love transformers']))
for i,x in z.items():
    print(f"{i} -> { len(x) if isinstance(x, list) else x.shape }")

Toh, BERT layer kai useful results return karti hai:
* `pooled_output` poore sequence ke saare tokens ka average nikaalne ka result hai. Ise poore network ka ek intelligent semantic embedding maana ja sakta hai. Yeh hamare pichhle model mein `GlobalAveragePooling1D` layer ke output ke barabar hai.
* `sequence_output` aakhri transformer layer ka output hai (jo hamare upar diye gaye model mein `TransformerBlock` ke output ke anuroop hai).
* `encoder_outputs` saari transformer layers ke outputs hain. Kyunki humne 4-layer BERT model load kiya hai (jaisa ki aap shayad naam se andaaza laga sakte hain, jisme `4_H` shamil hai), isme 4 tensors hain. Aakhri tensor `sequence_output` ke same hai.

Ab hum end-to-end classification model define karenge. Hum *functional model definition* use karenge, jisme hum model ka input define karenge aur phir uske output ko calculate karne ke liye expressions ki ek chain denge. Hum BERT model ke weights ko train nahi karenge aur sirf final classifier ko train karenge:

In [ ]:
inp = keras.Input(shape=(),dtype=tf.string)
x = vectorizer(inp)
x = bert(x)
x = keras.layers.Dropout(0.1)(x['pooled_output'])
out = keras.layers.Dense(4,activation='softmax')(x)
model = keras.models.Model(inp,out)
bert.trainable = False
model.summary()

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(128),validation_data=ds_test.map(tupelize).batch(128))

Halaanki trainable parameters bahut kam hain, process kaafi dheemi hai kyunki BERT feature extractor computationally heavy hai. Lagta hai ki hum reasonable accuracy hasil nahi kar paaye, ya toh training ki kami ki wajah se, ya model parameters ki kami ki wajah se.

Chalo BERT weights ko unfreeze karte hain aur ise bhi train karte hain. Iske liye bahut chhoti learning rate chahiye, aur saath hi **warmup** ke saath ek zyada careful training strategy chahiye, jisme **AdamW** optimizer use hota hai. Hum `tf-models-official` package use karke optimizer banayenge:

In [ ]:
from official.nlp import optimization 
bert.trainable=True
model.summary()
epochs = 3
opt = optimization.create_optimizer(
    init_lr=3e-5,
    num_train_steps=epochs*len(ds_train),
    num_warmup_steps=0.1*epochs*len(ds_train),
    optimizer_type='adamw')

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer=opt)
model.fit(ds_train.map(tupelize).batch(128),validation_data=ds_test.map(tupelize).batch(128))

Jaisa ki aap dekh sakte hain, training kaafi dheemi gati se hoti hai - lekin aap kuch epochs (5-10) ke liye model ko train karne ki koshish kar sakte hain aur dekh sakte hain ki kya aap pehle use kiye gaye tarikon ke muqabale best result hasil kar sakte hain.

## Huggingface Transformers library

Transformer models use karne ka ek aur bahut common (aur thoda simple) tarika [HuggingFace package](https://github.com/huggingface/) hai, jo alag-alag NLP tasks ke liye simple building blocks provide karta hai. Yeh Tensorflow aur PyTorch, ek aur bahut popular neural network framework, dono ke liye available hai.

> **Note**: Agar aapko yeh dekhne mein interest nahi hai ki Transformers library kaise kaam karti hai - toh aap is notebook ke end tak ja sakte hain, kyunki aap upar kiye gaye kaam se kuch bhi fundamentally alag nahi dekhenge. Hum BERT model ko train karne ke wahi steps repeat karenge, lekin ek alag library aur kaafi bade model ke saath. Isliye, process mein kuch lambi training shamil hai, toh aap sirf code ko dekh sakte hain.

Chalo dekhte hain ki hamari problem [Huggingface Transformers](http://huggingface.co) use karke kaise solve ki ja sakti hai.

Sabse pehle humein woh model chunna hoga jise hum use karne wale hain. Kuch built-in models ke alawa, Huggingface mein ek [online model repository](https://huggingface.co/models) bhi hai, jahan aapko community dwara banaye gaye aur bhi kai pre-trained models mil sakte hain. In saare models ko sirf model ka naam dekar load aur use kiya ja sakta hai. Model ke liye zaroori saari binary files automatically download ho jaayengi.

Kuch situations mein aapko apne khud ke models load karne pad sakte hain. Aise cases mein, aap woh directory specify kar sakte hain jisme saari related files maujood hon, jaise tokenizer ke parameters, `config.json` file jisme model parameters hon, binary weights, wagairah.

Model ke naam se, hum model aur tokenizer dono ko instantiate kar sakte hain. Chalo tokenizer se shuru karte hain:

In [ ]:
import transformers

# To load the model from Internet repository using model name. 
# Use this if you are running from your own copy of the notebooks
bert_model = 'bert-base-uncased' 

# To load the model from the directory on disk. Use this for Microsoft Learn module, because we have
# prepared all required files for you.
#bert_model = './bert'

tokenizer = transformers.BertTokenizer.from_pretrained(bert_model)

MAX_SEQ_LEN = 128
PAD_INDEX = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
UNK_INDEX = tokenizer.convert_tokens_to_ids(tokenizer.unk_token)

`tokenizer` object mein `encode` function hota hai jise seedhe text ko encode karne ke liye use kiya ja sakta hai:

In [ ]:
tokenizer.encode('Tensorflow is a great framework for NLP')

Hum tokenizer ka use sequence ko is tarah encode karne ke liye bhi kar sakte hain jo model ko pass karne ke liye suitable ho, jaise `token_ids`, `input_mask` fields wagairah shamil karna. Hum yeh bhi specify kar sakte hain ki humein Tensorflow tensors chahiye, iske liye `return_tensors='tf'` argument provide kar sakte hain:

In [ ]:
tokenizer(['Hello, there'],return_tensors='tf')

Hamare case mein, hum ek pre-trained BERT model use karenge jise `bert-base-uncased` kehte hain. *Uncased* ka matlab hai ki yeh model case-sensitive nahi hai.

Model ko train karte time, humein tokenized sequence ko input ke roop mein provide karna hota hai, aur isliye hum data processing pipeline design karenge. Kyunki `tokenizer.encode` ek Python function hai, hum ise pichhle unit ki tarah hi use karenge, jisme ise `py_function` use karke call kiya jaayega:

In [ ]:
def process(x):
    return tokenizer.encode(x.numpy().decode('utf-8'),return_tensors='tf',padding='max_length',max_length=MAX_SEQ_LEN,truncation=True)[0]

def process_fn(x):
    s = x['title']+' '+x['description']
    e = tf.py_function(process,inp=[s],Tout=(tf.int32))
    e.set_shape(MAX_SEQ_LEN)
    return e,x['label']

Ab hum `BertForSequenceClassification` package use karke actual model load kar sakte hain. Yeh ensure karta hai ki hamare model mein classification ke liye zaroori structure pehle se hai, jisme final classifier bhi shamil hai. Aapko ek warning message dikhega jisme kaha jaayega ki final classifier ke weights initialize nahi kiye gaye hain, aur model ko pre-training ki zarurat hogi - yeh bilkul theek hai, kyunki yahi hum karne wale hain!

In [ ]:
model = transformers.TFBertForSequenceClassification.from_pretrained(bert_model,num_labels=4,output_attentions=False)

In [ ]:
model.summary()

Jaisa ki aap `summary()` se dekh sakte hain, model mein lagbhag 110 million parameters hain! Shayad, agar hum relatively chhote dataset par simple classification task karna chahte hain, toh hum BERT base layer ko train nahi karna chahenge:

In [ ]:
model.layers[0].trainable = False
model.summary()

Ab hum training shuru karne ke liye taiyaar hain!

> **Note**: Full-scale BERT model ki training bahut time lene wali ho sakti hai! Isliye hum ise sirf pehle 32 batches ke liye train karenge. Yeh sirf yeh dikhane ke liye hai ki model training kaise set up ki jaati hai. Agar aap full-scale training try karne mein interested hain - toh bas `steps_per_epoch` aur `validation_steps` parameters hata dein, aur intezaar karne ke liye taiyaar rahein!

In [ ]:
model.compile('adam','sparse_categorical_crossentropy',['acc'])
tf.get_logger().setLevel('ERROR')
model.fit(ds_train.map(process_fn).batch(32),validation_data=ds_test.map(process_fn).batch(32),steps_per_epoch=32,validation_steps=2)

Agar aap iterations ki sankhya badhate hain, kaafi der tak wait karte hain, aur kai epochs tak training karte hain, toh aap umeed kar sakte hain ki BERT classification humein best accuracy dega! Iska kaaran yeh hai ki BERT pehle se hi language ki structure ko kaafi achhe se samajhta hai, aur humein sirf final classifier ko fine-tune karna hota hai. Lekin, kyunki BERT ek bada model hai, poori training process kaafi time leti hai aur iske liye serious computational power chahiye hoti hai! (GPU, aur preferably ek se zyada).

> **Note:** Hamare example mein, humne sabse chhote pre-trained BERT models mein se ek use kiya hai. Bade models available hain jo shayad behtar results de sakte hain.

## Key Takeaways

Is unit mein, humne haal hi ke **Transformers** par based model architectures dekhe hain. Humne inhein apne text classification task ke liye apply kiya hai, lekin isi tarah, BERT model ka use entity extraction, question answering aur doosre NLP tasks ke liye bhi kiya ja sakta hai.

Transformer models NLP mein abhi sabse advanced technology ko represent karte hain, aur zyadatar cases mein, yeh woh pehla solution hona chahiye jiske saath aap custom NLP solutions implement karte time experiment karna shuru karein. Lekin, agar aap advanced neural models banana chahte hain, toh is module mein discuss kiye gaye recurrent neural networks ke fundamental principles ko samajhna bahut zaroori hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->